# Day 069 — Exercise 4: Retry + Validation

**What you'll build:** `safe_extract` (retry wrapper) and `validate_extraction` (Pydantic conformance check).

**Why it matters:** Production vision LLM calls are non-deterministic — a prompt that fails once often succeeds on the next attempt. `safe_extract` handles this gracefully, and `validate_extraction` gives callers a clean (ok, result) tuple instead of forcing them to catch Pydantic exceptions everywhere.

In [ ]:
import io
import re
import json
import base64
from pydantic import BaseModel

def image_to_base64(img, format='PNG'):
    buf = io.BytesIO()
    out = img
    if format.upper() in ('JPEG', 'JPG') and img.mode in ('RGBA', 'P'):
        out = img.convert('RGB')
    out.save(buf, format=format)
    return base64.b64encode(buf.getvalue()).decode()

def build_extraction_prompt(schema_cls):
    schema_json = json.dumps(schema_cls.model_json_schema(), indent=2)
    return (
        'Extract structured data from this image and return ONLY valid JSON '
        'matching this schema exactly. Do not include any explanation, '
        'markdown, or code blocks.\n\n'
        f'Schema:\n{schema_json}\n\n'
        'Return ONLY the JSON object, nothing else.'
    )

def strip_json_from_response(response):
    block = re.search(r'```(?:json)?\s*([\s\S]*?)```', response)
    if block:
        return block.group(1).strip()
    obj = re.search(r'\{[\s\S]*\}', response)
    if obj:
        return obj.group(0).strip()
    raise ValueError(f'No JSON found: {response[:200]!r}')

def extract_from_image(img_b64, schema_cls, describe_fn=None):
    prompt = build_extraction_prompt(schema_cls)
    if describe_fn is not None:
        response = describe_fn(img_b64, prompt)
    else:
        import ollama
        resp = ollama.chat(
            model='llava',
            messages=[{'role': 'user', 'content': prompt, 'images': [img_b64]}]
        )
        response = resp['message']['content']
    raw = strip_json_from_response(response)
    return json.loads(raw)

def safe_extract(img_b64, schema_cls, describe_fn=None, retries=2):
    for attempt in range(retries + 1):
        try:
            return extract_from_image(img_b64, schema_cls, describe_fn=describe_fn)
        except (ValueError, json.JSONDecodeError):
            if attempt == retries:
                return None
    return None

def validate_extraction(data, schema_cls):
    try:
        model = schema_cls.model_validate(data)
        return (True, model)
    except Exception as exc:
        return (False, str(exc))

from pydantic import BaseModel

class _TestItem(BaseModel):
    name:  str
    value: float

class _TestSchema(BaseModel):
    title:   str
    amount:  float
    items:   list[_TestItem] = []
    note:    str = ''

from PIL import Image
_img = Image.new('RGB', (200, 100), 'white')
_img_b64 = image_to_base64(_img)


## Task

Implement both functions:

**`safe_extract(img_b64, schema_cls, describe_fn=None, retries=2)`:**
- `for attempt in range(retries + 1):`
- `try: return extract_from_image(...)`
- `except (ValueError, json.JSONDecodeError): if attempt == retries: return None`

**`validate_extraction(data, schema_cls) -> tuple`:**
- `try: return (True, schema_cls.model_validate(data))`
- `except Exception as exc: return (False, str(exc))`

## Your Implementation

In [ ]:
def safe_extract(img_b64: str, schema_cls,
                 describe_fn=None, retries: int = 2):
    """Extract JSON from an image with retry on parse failure.

    Args:
        img_b64:    base64-encoded image string
        schema_cls: Pydantic model class
        describe_fn: callable for testing
        retries:    number of extra attempts (total = retries + 1)
    Returns:
        dict on success, None if all attempts fail
    """
    raise NotImplementedError


def validate_extraction(data: dict, schema_cls) -> tuple:
    """Validate extracted data against a Pydantic schema.

    Args:
        data:       dict from extract_from_image or safe_extract
        schema_cls: Pydantic model class
    Returns:
        (True, validated_model) on success
        (False, error_message_str) on failure
    """
    raise NotImplementedError


In [ ]:
def safe_extract(img_b64: str, schema_cls,
                 describe_fn=None, retries: int = 2):
    for attempt in range(retries + 1):
        try:
            return extract_from_image(img_b64, schema_cls,
                                      describe_fn=describe_fn)
        except (ValueError, json.JSONDecodeError):
            if attempt == retries:
                return None
    return None


def validate_extraction(data: dict, schema_cls) -> tuple:
    try:
        model = schema_cls.model_validate(data)
        return (True, model)
    except Exception as exc:
        return (False, str(exc))


## Automated checks

In [ ]:
score, total = 0, 5
try:
    # safe_extract: good mock → returns dict
    _good = lambda b, p: '{"title": "Test", "amount": 42.0}'
    result = safe_extract(_img_b64, _TestSchema, describe_fn=_good)
    assert isinstance(result, dict), f"Expected dict, got {type(result)}"
    assert result['title'] == 'Test' and result['amount'] == 42.0
    score += 1; print("\u2705 safe_extract returns dict on success")

    # safe_extract: always-failing mock → returns None after retries
    _bad = lambda b, p: "no json here"
    result_none = safe_extract(_img_b64, _TestSchema, describe_fn=_bad, retries=1)
    assert result_none is None, f"Expected None, got {result_none}"
    score += 1; print("\u2705 safe_extract returns None after exhausting retries")

    # validate_extraction: valid data → (True, model)
    ok, model = validate_extraction({'title': 'Test', 'amount': 9.99}, _TestSchema)
    assert ok is True, f"Expected True, got {ok}"
    assert model.title == 'Test' and model.amount == 9.99
    score += 1; print("\u2705 validate_extraction returns (True, model) for valid data")

    # validate_extraction: missing required field → (False, error_str)
    ok2, err = validate_extraction({'title': 'Test'}, _TestSchema)
    assert ok2 is False, f"Expected False, got {ok2}"
    assert isinstance(err, str) and len(err) > 0
    score += 1; print("\u2705 validate_extraction returns (False, error_str) for invalid data")

    # validate_extraction: type coercion works (str "9.99" → float 9.99)
    ok3, model3 = validate_extraction({'title': 'Coerce', 'amount': '9.99'}, _TestSchema)
    assert ok3 is True and abs(model3.amount - 9.99) < 0.001
    score += 1; print("\u2705 validate_extraction coerces str to float")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def safe_extract(img_b64: str, schema_cls,
                 describe_fn=None, retries: int = 2):
    for attempt in range(retries + 1):
        try:
            return extract_from_image(img_b64, schema_cls,
                                      describe_fn=describe_fn)
        except (ValueError, json.JSONDecodeError):
            if attempt == retries:
                return None
    return None


def validate_extraction(data: dict, schema_cls) -> tuple:
    try:
        model = schema_cls.model_validate(data)
        return (True, model)
    except Exception as exc:
        return (False, str(exc))
```

**Why return `None` not raise on exhaustion?** `None` is a clear sentinel that the pipeline failed — different from an empty dict, which is a valid (though unusual) schema response. The caller decides whether to log, use a default, or surface an error, without being forced to wrap `safe_extract` in another try/except.

</details>